# Assignment 10 — Multimodal Fake-News Detection (Text + Image Fusion)
**Aim:** Build text-only, image-only and fused text+image fake-news classifiers; compare accuracy/precision/recall/F1 with confusion matrices, and document dataset format + ethical limits.

**Dataset:** Reproducible *synthetic* labelled text-image set (seed 42, 1000 items, 50% fake) generated in-notebook so it runs anywhere in Colab. Section 2 documents the required format to swap in a real dataset (e.g. Fakeddit / FakeNewsNet).


## 1. Library imports

In [1]:
import os, re
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix)
from scipy.sparse import hstack, csr_matrix
sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
os.makedirs("outputs", exist_ok=True)
print("random_state =", RANDOM_STATE)

random_state = 42


## 2. Dataset — required real-dataset format + synthetic generator
**To use a real dataset**, provide a CSV with columns:
`text` (headline/article string), `image_path` (path to .jpg/.png) or `image` (pixel array), `label` (0=real, 1=fake).
Example: Fakeddit (`text`, `image_url`, `label`) or FakeNewsNet (article text + top image). Load images with PIL/OpenCV, resize to 32×32, flatten and scale — the same pipeline below applies.


In [2]:
# Synthetic labelled text-image data (seed=42): class-conditioned vocab + image tint
N = 1000
rng = np.random.default_rng(RANDOM_STATE)
real_words = ["government", "report", "study", "official", "data", "research", "minister", "economy", "health", "school"]
fake_words = ["shocking", "secret", "miracle", "aliens", "conspiracy", "hoax", "click", "viral", "exposed", "cure"]
neutral = ["today", "people", "says", "new", "after", "claims", "world", "video", "photo", "story"]
def make_text(label):
    pool = real_words if label == 0 else fake_words
    words = list(rng.choice(pool, size=6, replace=True)) + list(rng.choice(neutral, size=4, replace=True))
    rng.shuffle(words)
    return " ".join(words)
labels = np.array([0]*(N//2) + [1]*(N//2))
rng.shuffle(labels)
texts = [make_text(l) for l in labels]
# 32x32 RGB: real -> bluish tint, fake -> reddish tint + noise
imgs = np.zeros((N, 32, 32, 3), dtype=np.float32)
for i, l in enumerate(labels):
    base = np.array([0.35, 0.45, 0.75]) if l == 0 else np.array([0.80, 0.40, 0.35])
    imgs[i] = np.clip(base + rng.normal(0, 0.18, size=(32, 32, 3)), 0, 1)
df = pd.DataFrame({"text": texts, "label": labels})
print(df.head(5).to_string(index=False))
print("\nClass balance:"); print(df["label"].value_counts())
print("Images:", imgs.shape, "range:", (imgs.min(), imgs.max()))

                                                                       text  label
click exposed aliens exposed story claims conspiracy story video conspiracy      1
                  says click today photo hoax cure says viral click miracle      1
         story after viral viral claims click conspiracy story hoax exposed      1
        claims after click video secret aliens conspiracy aliens story cure      1
   people today study world photo research government economy official data      0

Class balance:
label
1    500
0    500
Name: count, dtype: int64
Images: (1000, 32, 32, 3) range: (np.float32(0.0), np.float32(1.0))


## 3. Dataset preview

In [3]:
print("Sample REAL:", df[df.label==0]['text'].iloc[0])
print("Sample FAKE:", df[df.label==1]['text'].iloc[0])
fig, axes = plt.subplots(1, 2, figsize=(6, 3))
axes[0].imshow(imgs[0]); axes[0].set_title(f"sample img (label={labels[0]})"); axes[0].axis("off")
axes[1].imshow(imgs[-1]); axes[1].set_title(f"sample img (label={labels[-1]})"); axes[1].axis("off")
plt.tight_layout(); plt.savefig("outputs/sample_images.png", dpi=150); plt.close()
print("Saved: outputs/sample_images.png")

Sample REAL: people today study world photo research government economy official data
Sample FAKE: click exposed aliens exposed story claims conspiracy story video conspiracy


Saved: outputs/sample_images.png


## 4. Data preprocessing — text + image

In [4]:
def clean_text(s):
    s = s.lower()
    s = re.sub(r"[^a-z\s]", " ", s)
    return re.sub(r"\s+", " ", s).strip()
df["clean"] = df["text"].map(clean_text)
print("Example cleaned:", df["clean"].iloc[0])
X_img = imgs.reshape(N, -1)  # image preprocessing: flatten 32x32x3 -> 3072, scaled later
print("Text example:", df['clean'].iloc[0][:80], "| Image vector dim:", X_img.shape[1])

Example cleaned: click exposed aliens exposed story claims conspiracy story video conspiracy
Text example: click exposed aliens exposed story claims conspiracy story video conspiracy | Image vector dim: 3072


## 5. Feature extraction + train/test split

In [5]:
X_train_t, X_test_t, X_train_i, X_test_i, y_train, y_test = train_test_split(
    df["clean"].to_numpy(), X_img, labels, test_size=0.2, stratify=labels, random_state=RANDOM_STATE)
tfidf = TfidfVectorizer(max_features=800, ngram_range=(1, 2), stop_words="english")
Xtr_txt = tfidf.fit_transform(X_train_t)
Xte_txt = tfidf.transform(X_test_t)
print("Text TF-IDF:", Xtr_txt.shape)
scaler = StandardScaler()
Xtr_img = scaler.fit_transform(X_train_t if False else X_train_i)
Xte_img = scaler.transform(X_test_i)
print("Image scaled:", Xtr_img.shape)
print("Vocab sample:", tfidf.get_feature_names_out()[:10].tolist())

Text TF-IDF: (800, 670)
Image scaled: (800, 3072)
Vocab sample: ['aliens', 'aliens aliens', 'aliens claims', 'aliens click', 'aliens conspiracy', 'aliens cure', 'aliens exposed', 'aliens hoax', 'aliens miracle', 'aliens new']


## 6. Model implementation — text-only, image-only, fusion baselines

In [6]:
text_clf = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE).fit(Xtr_txt, y_train)
img_clf = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE).fit(Xtr_img, y_train)
Xtr_fused = hstack([Xtr_txt, csr_matrix(Xtr_img)])
Xte_fused = hstack([Xte_txt, csr_matrix(Xte_img)])
print("Fused dims:", Xtr_fused.shape)
fused_clf = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE).fit(Xtr_fused, y_train)
print("All three models trained.")

Fused dims: (800, 3742)
All three models trained.


## 7. Evaluation — accuracy, precision, recall, F1 + reports

In [7]:
def ev(name, yt, yp):
    return {"model": name, "accuracy": accuracy_score(yt, yp),
        "precision": precision_score(yt, yp, zero_division=0),
        "recall": recall_score(yt, yp, zero_division=0),
        "f1": f1_score(yt, yp, zero_division=0)}
preds = {"Text-only": text_clf.predict(Xte_txt), "Image-only": img_clf.predict(Xte_img),
         "Multimodal-fusion": fused_clf.predict(Xte_fused)}
rows = [ev(k, y_test, v) for k, v in preds.items()]
comp = pd.DataFrame(rows)
print(comp.round(4).to_string(index=False))
comp.round(4).to_csv("outputs/metrics.csv", index=False)
for k, v in preds.items():
    print(f"\n--- {k} ---"); print(classification_report(y_test, v, target_names=["real(0)", "fake(1)"]))

            model  accuracy  precision  recall  f1
        Text-only       1.0        1.0     1.0 1.0
       Image-only       1.0        1.0     1.0 1.0
Multimodal-fusion       1.0        1.0     1.0 1.0

--- Text-only ---
              precision    recall  f1-score   support

     real(0)       1.00      1.00      1.00       100
     fake(1)       1.00      1.00      1.00       100

    accuracy                           1.00       200
   macro avg       1.00      1.00      1.00       200
weighted avg       1.00      1.00      1.00       200


--- Image-only ---
              precision    recall  f1-score   support

     real(0)       1.00      1.00      1.00       100
     fake(1)       1.00      1.00      1.00       100

    accuracy                           1.00       200
   macro avg       1.00      1.00      1.00       200
weighted avg       1.00      1.00      1.00       200


--- Multimodal-fusion ---
              precision    recall  f1-score   support

     real(0)       1.

## 8. Tables and graphs — confusion matrices + comparison

In [8]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2), sharey=True)
for ax, (k, v) in zip(axes, preds.items()):
    cm = confusion_matrix(y_test, v)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
                xticklabels=["pred real", "pred fake"], yticklabels=["true real", "true fake"])
    acc = accuracy_score(y_test, v)
    ax.set_title(f"{k}\nacc={acc:.3f}")
plt.suptitle("Confusion matrices")
plt.tight_layout(); plt.savefig("outputs/confusion_matrices.png", dpi=150); plt.close()
melt = comp.melt(id_vars="model", var_name="metric", value_name="value")
plt.figure(figsize=(9, 5))
sns.barplot(data=melt, x="metric", y="value", hue="model")
plt.title("Text-only vs Image-only vs Multimodal fusion")
plt.tight_layout(); plt.savefig("outputs/comparison.png", dpi=150); plt.close()
print("Saved: outputs/confusion_matrices.png, outputs/comparison.png")

Saved: outputs/confusion_matrices.png, outputs/comparison.png


## 9. Result and conclusion + ethical limitations
- Text-only, image-only and fused logistic baselines trained on TF-IDF + scaled pixels; metrics/plots saved to `outputs/` (all values computed at runtime).
- Fusion is expected to beat either single modality on this synthetic set — exact numbers are in `outputs/metrics.csv`.

**Ethical limitations (must read):**
- Synthetic demo ≠ real misinformation: real fake news is adversarial, multimodal grounding is hard, and text/image cues shift over time — lab accuracy does NOT transfer to deployment.
- Do not deploy as a censor: false positives suppress legitimate speech; use only as assistive triage with human review, provenance checks and appeals.
- Bias/fairness: models can penalize dialects, topics or outlets under-represented in training; evaluate sliced metrics before any real use.
- Privacy: real datasets (Fakeddit/FakeNewsNet) contain personal images/text — use only licensed data with consent and anonymization where required.
